# 🛠️ Citadel Validation Environment - Setup Center

## Overview

Run this notebook **before any of the 14 validation scenarios directly in this directory**. It prepares one shared Python environment for backend onboarding, access contracts, agent frameworks, model APIs, authentication, PII/Cosmos analysis, session affinity, alerting, publish-contract A2A tests, and the 10-turn Foundry HR web-search chat.

| Stage | Purpose |
|---|---|
| 0 | Create the repository-root `.venv-validation-local` and select its kernel |
| 1 | Verify Python, the environment, and the working directory |
| 2 | Resolve the complete dependency set without installing |
| 3 | Install all dependencies together into the active kernel |
| 4 | Restart, verify pinned versions, and smoke-test SDK imports |
| 5 | Check external tools and continue to the validation suite |

> **Scope:** Archived notebooks under `archive-01` and `archive-02`, and notebooks in other directories, are not covered. This notebook does not deploy Azure resources or require Azure credentials for package verification.

> **Dedicated environment:** Use `.venv-validation-local`, not an existing repository `.venv` used by other labs. Installing requirements does not remove old packages, which can leave unrelated framework conflicts. The old environment can remain unchanged.

See the [Setup Guide](notebook-setup-guide.md) for terminal-only setup and troubleshooting, and the [Testing & Validation Guide](README.md) for deployment prerequisites and execution order.

## Prerequisites

- **Python 3.11** (use the latest available 3.11 patch release), VS Code, and the Microsoft **Python** and **Jupyter** extensions.
- Access to your approved Python package index. Do not bypass TLS verification if your organization requires a proxy or private feed.
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) for the validation notebooks; [Azure Developer CLI](https://learn.microsoft.com/azure/developer/azure-developer-cli/install-azd) only when using `from_azd = True`.

### How versions are handled

[requirements.txt](requirements.txt) is the **single source of truth for direct dependencies**. Each is pinned with `==` to keep ordinary reruns from silently upgrading SDK APIs. Python 3.11 is the supported setup target; other Python minor versions are not validated by this workflow.

- Microsoft Agent Framework uses the **core**, **OpenAI**, and **A2A** distributions only, not the `agent-framework` metapackage that installs every provider. Their release numbers need not match; pip resolves their declared compatibility constraints together.
- `agent-framework-a2a==1.0.0b261002` is an intentional preview pin. An explicit preview pin allows that package without a global `--pre`, which could opt unrelated dependencies into previews.
- Foundry's `PromptAgentDefinition` and `WebSearchTool`, Agent Framework's `AgentSession` and `OpenAIChatClient.as_agent`, A2A, and telemetry APIs are checked below. No automatic fallback downgrade or runtime package upgrade is performed.
- **This is not a full lockfile:** transitive dependencies are resolved by pip and can change. After verification, optionally capture `pip freeze` for a same-platform snapshot. Do not substitute a snapshot from another OS or Python version.
- To update: edit the central pins as a compatible set, resolve and install in a fresh Python 3.11 environment, rerun the checks here and affected validation notebooks, then commit the requirements and related guidance together. Do not upgrade SDKs individually inside scenario notebooks.

<a id='0'></a>
### 0️⃣ Create the Environment and Select the Kernel

Open the **repository root** in VS Code. Run these commands in a **terminal**, not a notebook cell. If `.venv-validation-local` already exists, verify it is Python 3.11 instead of overwriting an environment used by another kernel. Do not install this suite into the older `.venv`.

**Windows PowerShell (repository root):**

```powershell
py -3.11 --version
py -3.11 -m venv .venv-validation-local
.\.venv-validation-local\Scripts\python.exe -m pip install --upgrade pip setuptools wheel
.\.venv-validation-local\Scripts\python.exe -m pip install ipykernel==7.3.0
```

**macOS / Linux (repository root):**

```bash
python3.11 --version
python3.11 -m venv .venv-validation-local
.venv-validation-local/bin/python -m pip install --upgrade pip setuptools wheel
.venv-validation-local/bin/python -m pip install ipykernel==7.3.0
```

The initial `ipykernel` install is the bootstrap needed to run this notebook; it is also pinned in the complete requirements. Activation is optional because the commands target `.venv-validation-local` explicitly.

In this notebook's upper-right kernel picker, choose **Select Another Kernel > Python Environments > the repository's `.venv-validation-local`**. Installing packages in a terminal does not switch an open notebook's kernel. Do not continue with a global Python kernel or the old `.venv`.

If the environment is not listed, run **Python: Select Interpreter** from the Command Palette, choose **Enter interpreter path**, and select `.venv-validation-local\Scripts\python.exe` on Windows or `.venv-validation-local/bin/python` on macOS/Linux. Then reopen the notebook kernel picker and select that environment.

<a id='1'></a>
### 1️⃣ Verify the Active Environment

This standard-library-only cell checks the actual kernel, locates this directory from either the repository root or `validation`, and sets the working directory expected by the scenario notebooks. It fails before installation if the Python version, kernel, or location is wrong.

> After the restart in step 4, **rerun this cell**, then jump directly to step 4's verification cell. Do not rerun the installation cells just to verify.

### 1️⃣ Verify the Active Environment

Verify Python 3.11, the dedicated repository kernel, requirements path, and pip before installing dependencies.

**Input:** `repository`

**Output:** A validated dedicated interpreter and the pre-install dependency snapshot; incorrect kernels stop immediately.


In [ ]:
import sys
from pathlib import Path
repo_root = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "azure.yaml").is_file() and (p / "shared" / "utils.py").is_file()), None)
if repo_root is None:
    raise RuntimeError("Open this repository and run from its root or validation directory.")
sys.path.insert(0, str(repo_root / "shared"))
import utils
with utils.notebook_step('1️⃣ Verify the Active Environment', inputs={'repository': str(repo_root if 'repo_root' in globals() else root)}, expected='A validated dedicated interpreter and the pre-install dependency snapshot; incorrect kernels stop immediately.', namespace=globals()):
    if globals().get("_validation_restart_required", False):
        raise RuntimeError("STOP Run All: restart the kernel after dependency changes.")
    import importlib.metadata as md
    import os
    import subprocess
    import sys
    from pathlib import Path

    if sys.version_info[:2] != (3, 11):
        raise RuntimeError(f'Python 3.11 is required; active kernel is {sys.version.split()[0]}. See step 0.')

    current_dir = Path.cwd().resolve()
    validation_dir = current_dir if current_dir.name == 'validation' else current_dir / 'validation'
    requirements_path = validation_dir / 'requirements.txt'
    repo_root = validation_dir.parent
    if not requirements_path.is_file() or not (repo_root / 'shared' / 'utils.py').is_file():
        raise RuntimeError('Open this notebook from the repository root or validation directory.')
    if sys.prefix == sys.base_prefix or Path(sys.prefix).resolve() != (repo_root / '.venv-validation-local').resolve():
        raise RuntimeError(f'Select the repository .venv-validation-local kernel first. Active interpreter: {sys.executable}')

    os.chdir(validation_dir)
    pip_command = [sys.executable, '-m', 'pip', '--disable-pip-version-check']
    utils.format_output('1️⃣ Verify the Active Environment', {'value_1': 'Python:', 'value_2': sys.version.split()[0]}, status='info', part='RESULT')
    utils.format_output('1️⃣ Verify the Active Environment', {'value_1': 'Interpreter:', 'value_2': sys.executable}, status='info', part='RESULT')
    utils.format_output('1️⃣ Verify the Active Environment', {'value_1': 'Working directory:', 'value_2': Path.cwd()}, status='info', part='RESULT')
    utils.format_output('1️⃣ Verify the Active Environment', {'value_1': 'Requirements:', 'requirements_path': requirements_path}, status='info', part='RESULT')
    subprocess.run([*pip_command, '--version'], check=True)
    utils.format_output('1️⃣ Verify the Active Environment', {'value_1': 'Environment preflight: OK'}, status='info', part='RESULT')
    dependency_versions_before_install = {distribution.metadata['Name'].lower(): distribution.version
        for distribution in md.distributions() if distribution.metadata.get('Name')}


<a id='2'></a>
### 2️⃣ Resolve All Dependencies

Ask pip to resolve the **whole set in one transaction**. `--ignore-installed` ensures that an already-populated environment cannot hide a missing package or incompatible pin. This may download package metadata/wheels, but does not install packages or test native builds.

Any nonzero pip exit stops the cell with its original output. Resolve index, network, Python-version, or dependency errors before continuing; do not work around them with `--no-deps` or by installing packages one at a time.

### 2️⃣ Resolve All Dependencies

Resolve every pinned requirement together in a dry run, without modifying the environment.

**Input:** `pip_command`, `requirements_path`

**Output:** A successful complete dependency resolution, or a raised resolver error before installation.


In [ ]:
with utils.notebook_step('2️⃣ Resolve All Dependencies', inputs={'pip_command': globals().get('pip_command'), 'requirements_path': globals().get('requirements_path')}, expected='A successful complete dependency resolution, or a raised resolver error before installation.', namespace=globals()):
    subprocess.run(
        [*pip_command, 'install', '--dry-run', '--ignore-installed', '-r', str(requirements_path)],
        check=True,
    )
    utils.format_output('2️⃣ Resolve All Dependencies', {'value_1': 'Complete dependency resolution: OK. Continue to installation.'}, status='info', part='RESULT')


<a id='3'></a>
### 3️⃣ Install the Shared Dependencies

Install into **this kernel's interpreter**, never a bare `pip` or `--user` target. This includes optional scenario packages such as MSAL, Key Vault, Cosmos DB, and A2A so no later scenario needs to install its own dependencies.

There is deliberately no `--upgrade` or global `--pre`: exact pins are installed when missing or different, and already-satisfied versions remain unchanged. Pip's full output is kept for troubleshooting. A failure is not reported as success.

> An installation is not atomic. If pip fails, do not proceed to validation notebooks; fix the reported error and rerun, or create a fresh environment as described in the setup guide.

### 3️⃣ Install the Shared Dependencies

Install the resolved requirements and compare package versions against this kernel's pre-install snapshot.

**Input:** `dependency_versions_before_install`, `distribution`, `pip_command`, `requirements_path`

**Output:** Unchanged packages allow verification. Changed packages mark this kernel restart-required and stop Run All.


In [ ]:
with utils.notebook_step('3️⃣ Install the Shared Dependencies', inputs={'dependency_versions_before_install': globals().get('dependency_versions_before_install'), 'distribution': globals().get('distribution'), 'pip_command': globals().get('pip_command'), 'requirements_path': globals().get('requirements_path')}, expected='Unchanged packages allow verification. Changed packages mark this kernel restart-required and stop Run All.', namespace=globals()):
    if globals().get("_validation_restart_required", False):
        raise RuntimeError("STOP Run All: restart the kernel after dependency changes.")
    subprocess.run([*pip_command, 'install', '-r', str(requirements_path)], check=True)
    dependency_versions_after_install = {distribution.metadata['Name'].lower(): distribution.version
        for distribution in md.distributions() if distribution.metadata.get('Name')}
    if dependency_versions_after_install != dependency_versions_before_install:
        _validation_restart_required = True
        raise RuntimeError("Dependencies changed. STOP Run All: restart the kernel, then rerun preflight and verification.")
    utils.format_output('3️⃣ Install the Shared Dependencies', {'value_1': "Dependencies are unchanged; verification can continue in this kernel."}, status='info', part='RESULT')


<a id='4'></a>
### 4️⃣ Restart and Verify Versions / SDK APIs

If installation changed any package, **restart the kernel**, rerun **step 1 only**, then run the cell below. Run All deliberately raises at installation after dependency changes; rerunning without restarting cannot bypass that stop. When requirements are already unchanged, Run All can safely continue into verification.

Verification checks both installed package metadata **and actual imports/API symbols**, including the local shared helpers. `pip check` alone can pass when a required package was never installed. Its output is captured explicitly so a dependency conflict is visible in notebook output and in the raised error, rather than only a `CalledProcessError`. These checks do not call Azure or any model endpoint.

### 4️⃣ Restart and Verify Versions / SDK APIs

Check installed pins, dependency consistency, and the actual SDK symbols used by the scenario notebooks.

**Input:** `pip_command`, `requirements_path`

**Output:** Installed-version and import/API results; any mismatch raises rather than claiming setup success.


In [ ]:
with utils.notebook_step('4️⃣ Restart and Verify Versions / SDK APIs', inputs={'pip_command': globals().get('pip_command'), 'requirements_path': globals().get('requirements_path')}, expected='Installed-version and import/API results; any mismatch raises rather than claiming setup success.', namespace=globals()):
    if globals().get("_validation_restart_required", False):
        raise RuntimeError("STOP verification: restart the kernel after dependency changes.")
    import importlib
    import importlib.metadata as md
    from packaging.requirements import Requirement

    pip_check = subprocess.run(
        [*pip_command, 'check'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace', check=False,
    )
    pip_check_output = pip_check.stdout.strip()
    if pip_check_output:
        utils.format_output('4️⃣ Restart and Verify Versions / SDK APIs', {'pip_check_output': pip_check_output}, status='info', part='RESULT')
    if pip_check.returncode != 0:
        raise RuntimeError(
            f'pip check failed (exit code {pip_check.returncode}) for {sys.executable}:\n'
            f'{pip_check_output or "pip returned no diagnostic output."}\n\n'
            'Installing requirements does not remove previously installed packages. '
            'Select the dedicated validation environment created in step 0. '
            'If that environment also contains incompatible extras, recreate only that environment; '
            'do not downgrade the shared pins to satisfy unrelated frameworks.'
        )
    failures = []
    for line in requirements_path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        requirement = Requirement(line)
        if requirement.marker and not requirement.marker.evaluate():
            continue
        try:
            installed_version = md.version(requirement.name)
        except md.PackageNotFoundError:
            failures.append(f'{requirement.name}: missing')
            continue
        utils.format_output('4️⃣ Restart and Verify Versions / SDK APIs', {'value_1': f'{requirement.name}: {installed_version}'}, status='info', part='RESULT')
        if not requirement.specifier.contains(installed_version, prereleases=True):
            failures.append(f'{requirement.name}: expected {requirement.specifier}, found {installed_version}')

    shared_dir = str(repo_root / 'shared')
    if shared_dir not in sys.path:
        sys.path.insert(1, shared_dir)
    import_checks = {
        'IPython.display': ['display', 'Markdown'],
        'requests': ['Session'],
        'httpx': ['Client', 'AsyncClient'],
        'numpy': ['array'],
        'pandas': ['DataFrame'],
        'matplotlib.pyplot': ['subplots'],
        'matplotlib.backends': ['backend_registry'],
        'msal': ['PublicClientApplication'],
        'nest_asyncio': ['apply'],
        'azure.identity': ['DefaultAzureCredential'],
        'azure.mgmt.apimanagement': ['ApiManagementClient'],
        'azure.keyvault.secrets': ['SecretClient'],
        'azure.cosmos': ['CosmosClient'],
        'azure.ai.projects': ['AIProjectClient'],
        'azure.ai.projects.models': ['PromptAgentDefinition', 'WebSearchTool'],
        'openai': ['OpenAI', 'AzureOpenAI'],
        'agent_framework': ['AgentSession'],
        'agent_framework.openai': ['OpenAIChatClient.as_agent'],
        'agent_framework.a2a': ['A2AAgent'],
        'a2a.client': ['A2ACardResolver'],
        'opentelemetry.semconv_ai': ['SpanAttributes'],
        'langchain': [],
        'langchain_openai': ['ChatOpenAI'],
        'utils': ['print_ok'],
        'apimtools': ['APIMClientTool'],
    }
    for module_name, symbols in import_checks.items():
        try:
            module = importlib.import_module(module_name)
            for symbol in symbols:
                value = module
                for attribute in symbol.split('.'):
                    value = getattr(value, attribute)
        except (ImportError, AttributeError) as error:
            failures.append(f'{module_name}: {error}')
        else:
            utils.format_output('4️⃣ Restart and Verify Versions / SDK APIs', {'value_1': f'Import/API check: {module_name}: OK'}, status='info', part='RESULT')

    if failures:
        raise RuntimeError('Environment verification failed:\n- ' + '\n- '.join(failures))
    utils.format_output('4️⃣ Restart and Verify Versions / SDK APIs', {'value_1': 'Shared validation environment: OK. No Azure calls were made.'}, status='info', part='RESULT')


<a id='5'></a>
### 5️⃣ Verify External Tools and Continue

Azure CLI and Azure Developer CLI are **system tools**, not packages in `requirements.txt`. The next cell checks discovery only; it does not sign in or change your subscription. `azd` is optional for manual notebook configuration.

### 5️⃣ Verify External Tools and Continue

Locate Azure CLI and optional azd; authentication and subscription selection remain explicit terminal prerequisites.

**Input:** The initialized configuration and the explicit literals in this cell.

**Output:** Executable paths and the next authentication action; no Azure resource operation is performed.


In [ ]:
with utils.notebook_step('5️⃣ Verify External Tools and Continue', inputs={}, expected='Executable paths and the next authentication action; no Azure resource operation is performed.', namespace=globals()):
    import shutil

    az_path = shutil.which('az')
    azd_path = shutil.which('azd')
    if az_path is None:
        raise RuntimeError('Azure CLI was not found on PATH. Install it, then reopen VS Code and restart the kernel.')
    utils.format_output('5️⃣ Verify External Tools and Continue', {'value_1': 'Azure CLI:', 'az_path': az_path}, status='info', part='RESULT')
    utils.format_output('5️⃣ Verify External Tools and Continue', {'value_1': 'Azure Developer CLI:', 'value_2': azd_path or 'not found; required only for from_azd = True'}, status='info', part='RESULT')
    utils.format_output('5️⃣ Verify External Tools and Continue', {'value_1': 'Next: authenticate in a terminal and select your deployment subscription/environment.'}, status='info', part='RESULT')


## Continue to Validation

In a terminal, verify tools and authenticate:

```text
az --version
az login
az account show --output table
az account set --subscription <subscription-name-or-id>
```

For `from_azd = True`, also run `azd version`, `azd env list`, and `azd env select <environment-name>` from the repository root. Do not store secrets or authentication output in committed notebooks.

Open [Backend Onboarding](llm-backend-onboarding-runner.ipynb), then follow the [recommended execution order](README.md#recommended-execution-order). In **every notebook**, select the same `.venv-validation-local` kernel and confirm the working directory is `validation`; each notebook has its own kernel session. If necessary, use step 1 here as a preflight in that session before running its first cell. Agent Frameworks now consumes this environment instead of installing packages.

### Optional: Record the Resolved Versions

After successful verification, capture a same-OS/Python snapshot in the ignored `.venv-validation-local` directory. From the repository root on Windows:

```powershell
.\.venv-validation-local\Scripts\python.exe -m pip freeze > .\.venv-validation-local\validation.freeze.txt
```

On macOS/Linux, use `.venv-validation-local/bin/python -m pip freeze > .venv-validation-local/validation.freeze.txt`. Inspect snapshots before sharing: private package URLs can contain sensitive data. A snapshot records all installed transitive dependencies but is not a portable, hash-verified lockfile.

## Troubleshooting

| Symptom | Action |
|---|---|
| Wrong Python or global kernel | Return to step 0; select the repository's Python 3.11 `.venv-validation-local`. A venv cannot change its base Python in place. |
| Resolver / download / native build error | Read the first pip error and the full output; check Python, your approved index, proxy, and wheel availability. Do not continue after a failure. |
| `pip check` reports conflicts | Read the package names and constraints printed in the error. Select `.venv-validation-local`, not an older `.venv`; pip does not remove old frameworks when installing requirements. If needed, recreate only the dedicated environment; do not use `--no-deps` or downgrade pins to accommodate unrelated frameworks. |
| Imports fail after installation | Restart, rerun step 1, then step 4; compare the printed interpreter with the terminal's `.venv-validation-local` executable. |
| `utils` / `apimtools` not found | Confirm the working directory is `validation` and the repository includes `shared`. |
| Azure authorization / model errors | Package readiness is separate from deployment, RBAC, subscriptions, network access, secrets, and model availability; follow the scenario's Azure prerequisites. |

For more detail, see the [Setup Guide](notebook-setup-guide.md). There is no cloud cleanup step: this notebook creates no Azure resources.